In [ ]:
%pip install rouge_score bert_score evaluate
from datasets import load_dataset, Dataset
import pandas as pd
import numpy as np
import random as rd
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, Trainer, TrainingArguments, AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
from tqdm import tqdm
import evaluate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from peft import LoraConfig, get_peft_model, TaskType
from evaluate import load
from sklearn.model_selection import train_test_split
import math

In [ ]:
import torch
import math
import evaluate
from tqdm import tqdm
import re

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)  # Collapse multiple spaces
    return text

def generate_answer(model, tokenizer, question, context, max_length=512):
    # Prompt in stile naturale adatto a GPT-2
    prompt = f"{context}\n\nQuestion: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            num_beams=4,
            early_stopping=True
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Estrai solo la parte dopo "Answer:"
    if "Answer:" in decoded:
        return decoded.split("Answer:")[-1].strip()
    else:
        return decoded.strip()

from sentence_transformers import SentenceTransformer, util
import torch
import evaluate
import math
from tqdm import tqdm

def evaluate_model(model, tokenizer, dataset, max_samples=5, print_examples=True):
    model.eval()
    dataset = dataset.select(range(min(len(dataset), max_samples)))
    model_output=[]
    
    for i, example in enumerate(tqdm(dataset, desc="Evaluating")):
        single_output = []
        question = example["question"]
        context_list = example["topk_documents"]
        context = "\n".join(context_list)
        reference = example["answer"]

        # PPL
        full_target_text = f"{context}\n\nQuestion: {question}\nAnswer: {reference}"
        encoding = tokenizer(full_target_text, return_tensors="pt", truncation=True, max_length=512)
        input_ids = encoding["input_ids"].to(model.device)
        attention_mask = encoding["attention_mask"].to(model.device)
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
            losses.append(outputs.loss.item())
        prediction = generate_answer(model, tokenizer, question, context)
        
        single_output.append(question)
        single_output.append(context)
        single_output.append(reference)
        single_output.append(prediction)
        
        if print_examples:
            print(f"\n--- Example {i+1} ---")
            print(f"Question:\n{question}")
            print(f"Context:\n{context}")
            print(f"Prediction:\n{prediction}")
            print(f"Reference:\n{reference}")

        model_output.append(single_output)

    return model_output

In [ ]:
import evaluate
import math
from sentence_transformers import SentenceTransformer, util

def evaluate_output(model_output, tokenizer, eval_loss, print_examples=True):
    bertscore = evaluate.load("bertscore")
    rouge = evaluate.load("rouge")
    em = evaluate.load("exact_match")
    bleu = evaluate.load("bleu")
    embed_model = SentenceTransformer("all-MiniLM-L6-v2")

    predictions = []
    references = []
    faithfulness_scores = []
    relevance_scores = []
    length_ratios = []
    diversity_set = set()

    for i, example in enumerate(model_output):
        question = example[0]
        context = example[1]  # single string with all docs concatenated
        reference = example[2]
        prediction = example[3]

        # Faithfulness: sim(context, prediction)
        doc_emb = embed_model.encode(context, convert_to_tensor=True)
        pred_emb = embed_model.encode(prediction, convert_to_tensor=True)
        faithfulness_scores.append(util.cos_sim(pred_emb, doc_emb).item())

        # Relevance: sim(question, prediction)
        question_emb = embed_model.encode(question, convert_to_tensor=True)
        relevance_scores.append(util.cos_sim(pred_emb, question_emb).item())

        # BLEU (calcolato in seguito)
        predictions.append(prediction)
        references.append(reference)

        # Token Length Ratio
        ref_tokens = tokenizer.encode(reference, add_special_tokens=False)
        pred_tokens = tokenizer.encode(prediction, add_special_tokens=False)
        length_ratio = len(pred_tokens) / len(ref_tokens) if len(ref_tokens) > 0 else 0
        length_ratios.append(length_ratio)

        # Diversity (vocab size / total tokens)
        diversity_set.update(pred_tokens)

        if print_examples:
            print(f"\n--- Example {i+1} ---")
            print(f"Question:\n{question}")
            print(f"Context:\n{context}")
            print(f"Prediction:\n{prediction}")
            print(f"Reference:\n{reference}")

    # Metriche
    bertscore_f1 = bertscore.compute(predictions=predictions, references=references, lang="en")["f1"]
    rouge_score = rouge.compute(predictions=predictions, references=references)
    em_score = em.compute(predictions=predictions, references=references)["exact_match"]
    bleu_score = bleu.compute(predictions=predictions, references=references)["bleu"]
    perplexity_score = math.exp(eval_loss)

    token_count = sum(len(tokenizer.encode(p, add_special_tokens=False)) for p in predictions)
    diversity = len(diversity_set) / token_count if token_count > 0 else 0

    return {
        "RougeL": rouge_score["rougeL"],
        "EM": em_score,
        "BERTScore_F1": sum(bertscore_f1) / len(bertscore_f1),
        "BLEU": bleu_score,
        "Perplexity": perplexity_score,
        "Faithfulness": sum(faithfulness_scores) / len(faithfulness_scores),
        "Relevance": sum(relevance_scores) / len(relevance_scores),
        "Token_Length_Ratio": sum(length_ratios) / len(length_ratios),
        "Diversity": diversity
    }
